# 3. Estudo de Viabilidade: Auditoria Histórica (Schema Drift)

Antes de conectarmos 15 anos de dados em um pipeline de Big Data, precisamos provar se os dados antigos (2009 a 2016) possuem compatibilidade de dicionário com o nosso padrão-ouro de 2023.

Este notebook varre as pastas de dados brutos e realiza uma **Leitura Cega** apenas do cabeçalho dos arquivos. O objetivo é identificar onde o INEP quebrou a estrutura de dados (Schema Drift) e quais variáveis sumiram ou mudaram de nome no passado.

In [1]:
import pandas as pd
import os
import glob
import warnings

warnings.filterwarnings('ignore')

# Nosso "Esquadrão de Elite" + Targets
features_oficiais = ['Q006', 'Q024', 'TP_ANO_CONCLUIU', 'TP_LINGUA', 'TP_SEXO', 'Q002', 'NU_NOTA_MT', 'NU_NOTA_LC']

print("Variáveis requeridas para harmonização:")
print(features_oficiais)

Variáveis requeridas para harmonização:
['Q006', 'Q024', 'TP_ANO_CONCLUIU', 'TP_LINGUA', 'TP_SEXO', 'Q002', 'NU_NOTA_MT', 'NU_NOTA_LC']


## 1. Scanner de Arquivos
Vamos procurar todos os anos disponíveis na pasta `data/raw/`.

In [4]:
raw_dir = '../data/raw'
# Encontrar todos os diretórios de anos
anos_dirs = sorted(glob.glob(os.path.join(raw_dir, 'enem_*')))

arquivos_csv = {}
for dir_path in anos_dirs:
    ano = os.path.basename(dir_path).split('_')[1]
    # Busca o CSV principal lá dentro (ignorando maiúsculas/minúsculas)
    csv_files = glob.glob(os.path.join(dir_path, '**', '*.csv'), recursive=True)
    # Filtra arquivos pequenos ou de dicionário (busca o principal)
    for f in csv_files:
        if os.path.basename(f).upper().startswith('MICRODADOS_ENEM'):
            arquivos_csv[ano] = f
            break

print(f"Encontrados arquivos CSV para {len(arquivos_csv)} anos:")
for ano, path in arquivos_csv.items():
    print(f"- {ano}: {os.path.basename(path)}")

Encontrados arquivos CSV para 15 anos:
- 2009: MICRODADOS_ENEM_2009.csv
- 2010: MICRODADOS_ENEM_2010.csv
- 2011: MICRODADOS_ENEM_2011.csv
- 2012: MICRODADOS_ENEM_2012.csv
- 2013: MICRODADOS_ENEM_2013.csv
- 2014: MICRODADOS_ENEM_2014.csv
- 2015: MICRODADOS_ENEM_2015.csv
- 2016: microdados_enem_2016.csv
- 2017: MICRODADOS_ENEM_2017.csv
- 2018: MICRODADOS_ENEM_2018.csv
- 2019: MICRODADOS_ENEM_2019.csv
- 2020: MICRODADOS_ENEM_2020.csv
- 2021: MICRODADOS_ENEM_2021.csv
- 2022: MICRODADOS_ENEM_2022.csv
- 2023: MICRODADOS_ENEM_2023.csv


## 2. Auditoria de Colunas (Schema Audit)
Vamos ler APENAS a primeira linha de cada CSV (super rápido, não trava a memória) para extrair as colunas e bater contra o nosso padrão-ouro.

In [5]:
resultados_auditoria = []

for ano in sorted(arquivos_csv.keys()):
    caminho = arquivos_csv[ano]
    try:
        # Lê apenas 5 linhas para capturar o cabeçalho
        df_amostra = pd.read_csv(caminho, sep=';', encoding='latin-1', nrows=5)
        colunas_ano = df_amostra.columns.tolist()
        
        ausentes = [feat for feat in features_oficiais if feat not in colunas_ano]
        status = "✅ PERFEITO" if not ausentes else "❌ DRIFT ENCONTRADO"
        
        resultados_auditoria.append({
            'Ano': ano,
            'Total Colunas': len(colunas_ano),
            'Status': status,
            'Variáveis Ausentes': ", ".join(ausentes) if ausentes else "Nenhuma"
        })
    except Exception as e:
        print(f"Erro ao ler o ano {ano}: {e}")

df_auditoria = pd.DataFrame(resultados_auditoria)
display(df_auditoria)

,Ano,Total Colunas,Status,Variáveis Ausentes
0,2009,339,❌ DRIFT ENCONTRADO,"Q006, Q024, TP_ANO_CONCLUIU, TP_LINGUA, Q002"
1,2010,105,❌ DRIFT ENCONTRADO,"Q006, Q024, TP_ANO_CONCLUIU, Q002"
2,2011,128,✅ PERFEITO,Nenhuma
3,2012,115,✅ PERFEITO,Nenhuma
4,2013,130,❌ DRIFT ENCONTRADO,TP_ANO_CONCLUIU
5,2014,130,✅ PERFEITO,Nenhuma
6,2015,105,✅ PERFEITO,Nenhuma
7,2016,105,✅ PERFEITO,Nenhuma
8,2017,78,✅ PERFEITO,Nenhuma
9,2018,78,✅ PERFEITO,Nenhuma


## 3. Matriz de Decisão (Keep/Drop)
Com base na tabela acima, defina quais anos serão eliminados da tese devido à alta complexidade de harmonização estatística. Anos com `❌ DRIFT ENCONTRADO` na Renda (`Q006`) ou no Computador (`Q024`) exigiriam metodologias de imputação retroativa não suportadas pela literatura do INEP. A recomendação acadêmica é excluí-los da base de treinamento oficial.